# DataLens on Colab: Agentar-Scale-SQL around the GRPO model

This notebook replicates **Agentar-Scale-SQL** (Wang et al., 2025, the top published system on the
[BIRD leaderboard](https://bird-bench.github.io/), 81.67% on the test set) with the 3B model trained in
`colab_train_and_eval.ipynb`. The paper scales test-time compute in three ways:

| Step | What happens | Paper's model | Here |
|---|---|---|---|
| 1. Task understanding | find the database values the question mentions; retrieve similar train questions | Gemini-2.5-Flash | Qwen2.5-Coder-3B-Instruct |
| 2a. Reasoning generator | 8 candidates from the RL-trained model | GRPO-trained OmniSQL-32B | **your GRPO 3B model** |
| 2b. ICL generator | 9 candidates: 3 prompt styles x 3 temperatures, few-shot | Gemini-2.5-Pro, GPT-5 | Qwen2.5-Coder-3B-Instruct |
| 2c. Iterative refinement | fix errors and empty results, revise one query per result group | Gemini-2.5-Pro | Qwen2.5-Coder-3B-Instruct |
| 3. Tournament selection | group by result, pairwise round-robin judged by a selector | GRPO-trained Qwen2.5-Coder-32B | GRPO-trained Qwen2.5-Coder-3B |

All roles share one vLLM engine: the base Qwen2.5-Coder-3B-Instruct, with your GRPO adapter and the
selector adapter applied per request. See `docs/AGENT.md` for details and deviations.

**Runtime:** a G4 (RTX PRO 6000, 96 GB) or an A100. Every 3B role fits either.

**Order:** sections 1-4 need no new training and give a first result with an untrained (zero-shot) selector.
Section 5 trains the selector as the paper does; section 6 reruns selection with it.

**If the runtime disconnects,** rerun the Setup cell, then the cell you were on. Each stage resumes from
the last saved chunk; indexes and runs live on Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup
Same as the training notebook, plus `sentence-transformers` for the example retrieval of step 1.
The value indexes and the example store go to Drive (`agent-index/`), so they are built once.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WORKDIR = "/content/drive/MyDrive/datalens"      # the same folder as the training notebook
BRANCH = "claude/upbeat-galileo-p9fyzh"          # the agent's branch; use "main" once it is merged

In [ ]:
%cd /content
!git clone -q -b {BRANCH} https://github.com/RohanSada/DataLens.git || (cd DataLens && git fetch -q && git checkout -q {BRANCH} && git pull -q)
%cd /content/DataLens
!pip install -q -e ".[train,eval,vllm,agent]" "transformers==5.13.1"
!pip uninstall -q -y torchao
!mkdir -p {WORKDIR}/runs {WORKDIR}/checkpoints {WORKDIR}/reports {WORKDIR}/agent-index .cache
!ln -sfn {WORKDIR}/runs runs && ln -sfn {WORKDIR}/checkpoints checkpoints && ln -sfn {WORKDIR}/reports reports
!ln -sfn {WORKDIR}/agent-index .cache/agent

In [ ]:
!python -c "import torch, vllm, sentence_transformers; print('GPU:', torch.cuda.is_available(), '| vllm', vllm.__version__)" 

## 1. Data and the GRPO adapter
Both BIRD splits are needed: train supplies the few-shot examples (and, in section 5, the selector's
training data). The agent loads the GRPO model as a LoRA adapter on the base model; it is the
`adapter_*` files that training saved next to `merged/`.

In [ ]:
!datalens data download --split dev --split train

In [ ]:
import json, pathlib
adapter = pathlib.Path("checkpoints/grpo-qwen2.5-coder-3b/adapter_config.json")
assert adapter.is_file(), "train the GRPO model first (colab_train_and_eval.ipynb, section 3)"
cfg = json.loads(adapter.read_text())
print("base:", cfg["base_model_name_or_path"], "| rank:", cfg["r"], "| modules:", cfg["target_modules"])

## 2. Offline preprocessing
The paper's offline step: an index of every short text value in each dev database (for "matched contents"),
and the few-shot example store: the base model writes a question *skeleton* for each of the 9,428 train
questions ("How many <COLUMN> in <YEAR>?"), which is embedded with all-MiniLM-L6-v2. Optional (the first
run builds both on demand), but it keeps the one-off cost out of the timed run.

In [ ]:
!datalens agent index configs/agent/agentar-qwen2.5-coder-3b-zeroshot.yaml

## 3. Agent with an untrained selector
Each stage runs in its own process, so GPU memory is released between them, and each can be rerun to resume.
`understand` extracts literals and skeletons, `reason` and `icl` generate the 8 + 9 candidates, `refine` runs
the fixer and reviser, and `select` runs the tournament (here judged by the untrained base model).

In [ ]:
for stage in ["understand", "reason", "icl", "refine", "select"]:
    !datalens agent run configs/agent/agentar-qwen2.5-coder-3b-zeroshot.yaml --stage {stage}

## 4. Score and analyse
`eval score` grades the agent like any other run: its answer is scored as the greedy answer, and the 17
generated candidates as samples (so "EX with SC" is a majority vote over them). `agent analyze` adds the
paper's ablation table, the generator comparison and the selector's pairwise accuracy.

In [ ]:
!datalens eval score runs/agentar-qwen2.5-coder-3b-zeroshot
!datalens agent analyze runs/agentar-qwen2.5-coder-3b-zeroshot

## 5. Train the selector (paper section 3.4.2)
1. Run the generation stages on 4,000 BIRD train questions (few-shot examples come from *other* train databases).
2. Keep questions whose candidates include both a correct and a wrong result, and make pairs: the correct
   query against the most-voted wrong one and against one more, in random order.
3. GRPO on those pairs with reward 1 for picking the correct one, same recipe as the generator.
   Checkpoints go to Drive, so rerunning the training cell resumes it.

In [ ]:
for stage in ["understand", "reason", "icl", "refine"]:
    !datalens agent run configs/agent/agentar-qwen2.5-coder-3b-train.yaml --stage {stage}

In [ ]:
!datalens agent selector-data runs/agentar-qwen2.5-coder-3b-train --out data/selector/bird-train-pairs.jsonl

In [ ]:
!datalens train grpo configs/train/grpo-selector-qwen2.5-coder-3b.yaml
# out of GPU memory on a 40 GB A100: add --set trainer.per_device_train_batch_size=2 --set trainer.gradient_accumulation_steps=32

## 6. Selection with the trained selector
The candidates are the same as in section 3, so copy those stages and rerun only `select`. The two runs then
differ in nothing but the selector, which makes the comparison exactly paired.

In [ ]:
import pathlib, shutil

src, dst = pathlib.Path("runs/agentar-qwen2.5-coder-3b-zeroshot"), pathlib.Path("runs/agentar-qwen2.5-coder-3b")
dst.mkdir(parents=True, exist_ok=True)
for name in ["understanding.jsonl", "reasoning.jsonl", "icl.jsonl", "refinement.jsonl"]:
    if not (dst / name).exists():
        shutil.copy(src / name, dst / name)

In [ ]:
!datalens agent run configs/agent/agentar-qwen2.5-coder-3b.yaml --stage select
!datalens eval score runs/agentar-qwen2.5-coder-3b
!datalens agent analyze runs/agentar-qwen2.5-coder-3b

## 7. Optional: the task-understanding ablation
Dropping task understanding changes the prompts, so unlike the other ablations it needs its own run.

In [ ]:
for stage in ["understand", "reason", "icl", "refine", "select"]:
    !datalens agent run configs/agent/agentar-qwen2.5-coder-3b.yaml --stage {stage} --run-name agentar-3b-no-understanding --set understanding.enabled=false
!datalens eval score runs/agentar-3b-no-understanding
!datalens agent analyze runs/agentar-qwen2.5-coder-3b --compare "w/o task understanding=runs/agentar-3b-no-understanding"

## 8. Report
The agent runs join the comparison table next to the base, SFT and GRPO models.

In [ ]:
import os
from IPython.display import Markdown, display

candidates = ["base-qwen2.5-coder-3b", "sft-qwen2.5-coder-3b", "grpo-qwen2.5-coder-3b",
              "agentar-qwen2.5-coder-3b-zeroshot", "agentar-qwen2.5-coder-3b",
              "frontier-claude-sonnet-5-5", "frontier-claude-opus-5-5"]
run_dirs = " ".join(f"runs/{name}" for name in candidates if os.path.isfile(f"runs/{name}/scored.jsonl"))
!datalens eval report {run_dirs} --out reports/bird-dev
display(Markdown(open("reports/bird-dev/results.md").read()))
display(Markdown(open("runs/agentar-qwen2.5-coder-3b/analysis.md").read()))